# 004 — Enriquecimento Funcional dos Módulos WGCNA

Analisa os **4 módulos co-expressos significativamente associados ao tamanho de partícula** (padj < 0,10) identificados no notebook 003:

| Módulo | Correlação | Direção |
|--------|-----------|--------|
| ME_darkgrey | r = +0,63 | ↑ 1 µm |
| ME_tan | r = +0,59 | ↑ 1 µm |
| ME_linen | r = −0,58 | ↑ 100 nm |
| ME_darkred | r = −0,56 | ↑ 100 nm |

**Método:** Fisher's exact test local (`gseapy.enrich`) com background dos 12.174 genes filtrados.  
**Gene sets testados:** GO Biological Process 2023, GO Molecular Function 2023, KEGG 2021 Human.  
**Limiar de significância:** FDR (BH) < 0,05.

> ⚠️ Os módulos tan (9 genes) e linen (18 genes) são pequenos — é possível que não haja termos significativos.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import mygene
import numpy as np
import pandas as pd
import gseapy as gp

warnings.filterwarnings("ignore")

BASE_DIR = Path("../..").resolve()
DATA_INTERIM = BASE_DIR / "data" / "interim"
WGCNA_DIR = DATA_INTERIM / "wgcna"
ENRICHMENT_DIR = DATA_INTERIM / "enrichment"
ENRICHMENT_FIGURES_DIR = ENRICHMENT_DIR / "figures"

ENRICHMENT_DIR.mkdir(parents=True, exist_ok=True)
ENRICHMENT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Módulos significativos: padj < 0,10 para particle_size_um (decisão tomada no nb 003)
SIGNIFICANCE_COL = "significant_particle_size_um_padj010"
MODULE_PREFIX = "ME_"

GENE_SETS = [
    "GO_Biological_Process_2023",
    "GO_Molecular_Function_2023",
    "KEGG_2021_Human",
]
ENRICHMENT_PADJ_CUTOFF = 0.05
TOP_N_TERMS = 15

## 1. Módulos Significativos do WGCNA

In [ ]:
module_summary = pd.read_csv(WGCNA_DIR / "wgcna_module_summary.csv")
gene_modules   = pd.read_csv(WGCNA_DIR / "wgcna_gene_modules.csv")

sig_rows = module_summary[module_summary[SIGNIFICANCE_COL].astype(bool)]
sig_module_labels = sig_rows["module"].tolist()
sig_module_names  = [m.replace(MODULE_PREFIX, "") for m in sig_module_labels]

print("Módulos selecionados (α = 0,10 para particle_size_um):")
for label, name in zip(sig_module_labels, sig_module_names):
    row = sig_rows[sig_rows["module"] == label].iloc[0]
    n_genes = (gene_modules["module"] == name).sum()
    print(
        f"  {label:20s}  r={row['corr_particle_size_um']:+.3f}  "
        f"padj={row['padj_particle_size_um']:.4f}  n={n_genes} genes"
    )


## 2. Mapeamento Ensembl → Símbolo

O WGCNA usa Ensembl IDs. O GSEApy espera gene symbols. Consultamos a API mygene e cacheamos o resultado para evitar re-consultas.

In [ ]:
SYMBOL_CACHE_PATH = DATA_INTERIM / "mygene_symbol_cache.csv"

expression_filtered = pd.read_csv(DATA_INTERIM / "microplastic_expression_filtered.csv")
all_ensembl_ids = expression_filtered["gene_id"].tolist()

if SYMBOL_CACHE_PATH.exists():
    symbol_df = pd.read_csv(SYMBOL_CACHE_PATH, index_col="ensembl_id")
    print(f"Cache de símbolos carregado: {len(symbol_df)} genes")
else:
    print("Consultando mygene para símbolos (pode demorar ~30s)...")
    mg = mygene.MyGeneInfo()
    result = mg.querymany(
        all_ensembl_ids,
        scopes="ensembl.gene",
        fields="symbol",
        as_dataframe=True,
    )
    symbol_df = result[["symbol"]].copy()
    symbol_df.index.name = "ensembl_id"
    # Resolve duplicates: keep highest _score (already sorted by mygene)
    symbol_df = symbol_df[~symbol_df.index.duplicated(keep="first")]
    symbol_df.to_csv(SYMBOL_CACHE_PATH)
    print(f"Cache salvo em {SYMBOL_CACHE_PATH}: {len(symbol_df)} genes")

# Merge symbols into gene_modules
gene_modules_sym = gene_modules.merge(
    symbol_df.reset_index(),
    left_on="gene_id",
    right_on="ensembl_id",
    how="left",
)
n_missing = gene_modules_sym["symbol"].isna().sum()
print(f"Genes sem símbolo: {n_missing} ({n_missing / len(gene_modules_sym) * 100:.1f}%)")

## 3. Listas de Genes por Módulo e Background

In [ ]:
module_gene_lists = {}
for mod_name in sig_module_names:
    symbols = (
        gene_modules_sym.loc[gene_modules_sym["module"] == mod_name, "symbol"]
        .dropna()
        .unique()
        .tolist()
    )
    module_gene_lists[mod_name] = symbols
    print(f"  {MODULE_PREFIX}{mod_name}: {len(symbols)} genes com símbolo")

# Background: todos os 12.174 genes filtrados que têm símbolo
background_symbols = gene_modules_sym["symbol"].dropna().unique().tolist()
print(f"\nBackground: {len(background_symbols)} genes")

## 4. Enriquecimento Funcional

Usa `gseapy.enrich()` — Fisher's exact test local — com background customizado dos 12.174 genes filtrados.

In [ ]:
# Baixa bibliotecas uma vez (cacheadas pelo gseapy após o primeiro download)
print("Carregando bibliotecas de gene sets...")
libraries = {}
for gs_name in GENE_SETS:
    libraries[gs_name] = gp.get_library(gs_name, organism="Human")
    print(f"  {gs_name}: {len(libraries[gs_name])} termos")

In [ ]:
ENRICHMENT_CACHE = ENRICHMENT_DIR / "enrichment_combined.csv"

if ENRICHMENT_CACHE.exists():
    enrichment_combined = pd.read_csv(ENRICHMENT_CACHE)
    print(f"Cache de enriquecimento carregado: {len(enrichment_combined)} linhas")
else:
    all_results = []
    for mod_name in sig_module_names:
        gene_list = module_gene_lists[mod_name]
        print(f"
--- {MODULE_PREFIX}{mod_name} ({len(gene_list)} genes) ---")
        for gs_name, gene_set_lib in libraries.items():
            enr = gp.enrich(
                gene_list=gene_list,
                gene_sets=gene_set_lib,
                background=background_symbols,
                outdir=None,
                no_plot=True,
                cutoff=1.0,
            )
            df = enr.results.copy()
            df["module"]   = MODULE_PREFIX + mod_name
            df["gene_set"] = gs_name
            n_sig = (df["Adjusted P-value"] < ENRICHMENT_PADJ_CUTOFF).sum()
            print(f"  {gs_name}: {n_sig} termos significativos")
            all_results.append(df)

    enrichment_combined = pd.concat(all_results, ignore_index=True)
    enrichment_combined.to_csv(ENRICHMENT_CACHE, index=False)
    print(f"
Total testados: {len(enrichment_combined)}  "
          f"Significativos: {(enrichment_combined['Adjusted P-value'] < ENRICHMENT_PADJ_CUTOFF).sum()}")


## 5. Salvar Resultados

In [ ]:
# CSVs por módulo/gene_set
for (mod_label, gs_name), group in enrichment_combined.groupby(["module", "gene_set"]):
    mod_clean = mod_label.replace(MODULE_PREFIX, "")
    group.to_csv(ENRICHMENT_DIR / f"enrichment_{mod_clean}_{gs_name}.csv", index=False)
print(f"Arquivos salvos em {ENRICHMENT_DIR}")

# Resumo
sig_mask = enrichment_combined["Adjusted P-value"] < ENRICHMENT_PADJ_CUTOFF
summary = (
    enrichment_combined[sig_mask]
    .groupby(["module", "gene_set"])
    .size()
    .reset_index(name="n_significant")
)
print("
Resumo de termos significativos por módulo e gene set:")
print(summary.to_string(index=False))


## 6. Visualização — Barras Horizontais

In [ ]:
def plot_barh(df_sig, module_label, gs_name, top_n=TOP_N_TERMS, save_path=None):
    df_plot = df_sig.sort_values("Combined Score", ascending=False).head(top_n).copy()
    if df_plot.empty:
        print(f"  {module_label} × {gs_name}: sem termos significativos, plot pulado")
        return

    df_plot["-log10(padj)"] = -np.log10(df_plot["Adjusted P-value"].clip(lower=1e-300))
    df_plot["Term_short"]   = df_plot["Term"].str[:65]
    df_plot = df_plot.sort_values("Combined Score", ascending=True)  # topo = maior CS

    # Normalizar cor pelo range completo de df_sig, não só do top_n
    full_log10 = -np.log10(df_sig["Adjusted P-value"].clip(lower=1e-300))
    vmin, vmax = full_log10.min(), full_log10.max()
    if vmin == vmax:  # edge case: único valor de padj
        vmin, vmax = vmin - 0.5, vmax + 0.5
    norm   = plt.Normalize(vmin, vmax)
    colors = plt.cm.RdYlBu_r(norm(df_plot["-log10(padj)"].values))

    fig, ax = plt.subplots(figsize=(10, max(3, 0.42 * len(df_plot))))
    bars = ax.barh(
        df_plot["Term_short"], df_plot["Combined Score"],
        color=colors, edgecolor="white", linewidth=0.5, height=0.7,
    )

    x_max = df_plot["Combined Score"].max()
    for bar, overlap in zip(bars, df_plot["Overlap"]):
        ax.text(
            bar.get_width() + x_max * 0.01,
            bar.get_y() + bar.get_height() / 2,
            overlap, va="center", ha="left", fontsize=8, color="#555555",
        )

    sm = plt.cm.ScalarMappable(cmap="RdYlBu_r", norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="-log10(padj)", shrink=0.6, pad=0.02)

    ax.set_xlabel("Combined Score")
    ax.set_xlim(right=x_max * 1.18)
    gs_short = gs_name.replace("_2023", "").replace("_2021_Human", "").replace("_", " ")
    ax.set_title(f"{module_label}  —  {gs_short}", fontsize=10, fontweight="bold", pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Salvo: {save_path.name}")
    plt.show()
    plt.close()


In [ ]:
for (mod_label, gs_name), group in enrichment_combined.groupby(["module", "gene_set"]):
    sig = group[group["Adjusted P-value"] < ENRICHMENT_PADJ_CUTOFF]
    mod_clean = mod_label.replace(MODULE_PREFIX, "")
    save_path = ENRICHMENT_FIGURES_DIR / f"barh_{mod_clean}_{gs_name}.png"
    plot_barh(sig, mod_label, gs_name, save_path=save_path)